# Evaluate Trained Model — Small Reasoning LLM Lab

This notebook evaluates the **already-trained** `best.pt` checkpoint.
Do NOT retrain. Do NOT overwrite checkpoints.

What this notebook does:
1. Pull latest code (bug fixes)
2. Load `best.pt`
3. Re-run normal test benchmark (verify 17.8% result)
4. Run fixed generalization benchmark (Levels 1–5)
5. Compare all checkpoints on representative problems
6. Show actual model outputs including failures
7. Honest assessment of what the model learned

**Checkpoint path:** `experiments/results/colab_small_baseline/checkpoints/best.pt`

## 0. Pull Latest Code

In [ ]:
import os, sys

# If already in small-llm-lab directory, just pull
if os.path.exists('model') and os.path.exists('training'):
    print('Already in repo directory')
    !git pull origin main
elif os.path.exists('small-llm-lab'):
    os.chdir('small-llm-lab')
    !git pull origin main
else:
    !git clone https://github.com/sinor77/small-llm-lab.git
    os.chdir('small-llm-lab')

sys.path.insert(0, '.')
print(f'Working directory: {os.getcwd()}')
!git log --oneline -3

## 1. Setup

In [ ]:
import torch, json, os, time
from model.config import ModelConfig
from model.transformer import SmallTransformer
from model.tokenizer import BPETokenizer
from training.checkpoint import load_checkpoint
from data.generators.arithmetic import ArithmeticGenerator
from evaluation.benchmark import run_benchmark, run_generalization_benchmark
from evaluation.arithmetic import verify, _extract_answer
from inference.generate import (
    load_model_and_tokenizer, generate_answer,
    solve_problem, compare_checkpoints
)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
EXP_DIR    = 'experiments/results/colab_small_baseline'
CK_DIR     = os.path.join(EXP_DIR, 'checkpoints')
BEST_CK    = os.path.join(CK_DIR, 'best.pt')
TOK_DIR    = os.path.join(EXP_DIR, 'tokenizer')

# Verify all required files exist
required = [BEST_CK, os.path.join(TOK_DIR, 'tokenizer.json')]
for f in required:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {f}')

print(f'\nDevice: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Load best.pt once — reused throughout this notebook
model, tokenizer, ck = load_model_and_tokenizer(BEST_CK, device=DEVICE)
n_params = model.count_parameters()
step     = ck.get('global_step', '?')
cfg      = model.config

print('=== Model loaded ===')
print(f'  Parameters:   {n_params:,}')
print(f'  Training step: {step}')
print(f'  d_model={cfg.d_model}, n_layers={cfg.n_layers}, n_heads={cfg.n_heads}, d_ff={cfg.d_ff}')
print(f'  vocab_size={cfg.vocab_size}, max_seq_len={cfg.max_seq_len}')
print(f'  Tokenizer vocab: {tokenizer.vocab_size}')

# Generator matching the training config
generator = ArithmeticGenerator(seed=42, difficulty='medium')

## 2. Part 1 — Repository Status (code verification)

In [ ]:
import inspect
from evaluation import benchmark as bm_module
from inference import generate as gen_module

# 1. Confirm gen_level_acc is gone from benchmark.py
bm_src = inspect.getsource(bm_module)
has_dead_var = 'gen_level_acc[level]' in bm_src
print(f'gen_level_acc[level] in benchmark.py: {has_dead_var}  (expected: False)')

# 2. Confirm the three inference functions exist
for fn_name in ['solve_problem', 'interactive_inference', 'compare_checkpoints']:
    exists = hasattr(gen_module, fn_name)
    print(f'{fn_name} in generate.py: {exists}  (expected: True)')

# 3. Confirm no duplicate main() — check that each top-level function appears only once
gen_src = inspect.getsource(gen_module)
load_count = gen_src.count('def load_model_and_tokenizer')
main_count = gen_src.count('def main():')
print(f'load_model_and_tokenizer defined {load_count}x  (expected: 1)')
print(f'main() defined {main_count}x  (expected: 1)')

## 3. Part 2 — Generalization Benchmark (fixed)

In [ ]:
# Build exclusion set from the training data to guarantee zero overlap
print('Generating exclusion set from training data...')
train_ex, val_ex, test_ex = generator.generate_all_splits(
    n_train=20000, n_val=2000, n_test=1000
)
all_known = set(e.problem for e in train_ex + val_ex + test_ex)
print(f'Exclusion set: {len(all_known):,} unique problems')

In [ ]:
print('Running generalization benchmark (n_per_level=200)...')
t0 = time.time()

gen_results = run_generalization_benchmark(
    model=model,
    tokenizer=tokenizer,
    generator=generator,
    n_per_level=200,
    max_new_tokens=64,
    temperature=0.0,
    device=DEVICE,
    use_reasoning=True,
    exclude_problems=all_known,
)

elapsed = time.time() - t0

print(f'\nCompleted in {elapsed:.0f}s')
print()
print(f'{"Level":<8} {"Correct":>8} {"Total":>8} {"Accuracy":>10}')
print('-' * 38)

total_correct = 0
total_examples = 0
for level in sorted(gen_results.keys()):
    r = gen_results[level]
    print(f'{level:<8} {r["correct"]:>8} {r["total"]:>8} {r["accuracy"]:>10.1%}')
    total_correct  += r['correct']
    total_examples += r['total']

overall = total_correct / total_examples if total_examples > 0 else 0.0
print('-' * 38)
print(f'{"Overall":<8} {total_correct:>8} {total_examples:>8} {overall:>10.1%}')

# Save results
gen_out = os.path.join(EXP_DIR, 'generalization_fixed.json')
with open(gen_out, 'w') as f:
    json.dump(gen_results, f, indent=2)
print(f'\nSaved to: {gen_out}')

### Level definitions (what each level actually tests)
- **Level 1** — New values, same operation templates, same difficulty (medium). Minimum bar: did the model learn procedures or just specific number pairs?
- **Level 2** — Same families, hard difficulty (numbers 1–999 vs training's 1–100). Tests whether the model generalises to larger numbers.
- **Level 3** — Mixed families, medium difficulty. Consistency check across problem types.
- **Level 4** — Hard difficulty, multi_step + algebra_linear only. Longest reasoning chains.
- **Level 5** — All families, hard difficulty. Combined stress test.

## 4. Part 3 — Verify Normal Test Accuracy

In [ ]:
print('Running standard test benchmark (n=500, split=test)...')
t0 = time.time()

bench = run_benchmark(
    model=model,
    tokenizer=tokenizer,
    generator=generator,
    n_problems=500,
    split='test',
    max_new_tokens=64,
    temperature=0.0,
    use_reasoning=True,
    max_samples_to_save=50,
    device=DEVICE,
)

elapsed = time.time() - t0
print(f'\nCompleted in {elapsed:.0f}s')
print()
print(bench.summary_str())
print()
print(f'Previous result: 89/500 = 17.8%')
print(f'Current result:  {bench.correct}/{bench.total} = {bench.accuracy:.1%}')
if abs(bench.accuracy - 0.178) < 0.02:
    print('Result consistent with previous run (within 2%).')
else:
    print('NOTE: Result differs from previous by more than 2% -- check data generation.')

## 5. Part 4 — Checkpoint Progression

In [ ]:
# Representative problems — one per family, includes hard cases and failures
PROBE_PROBLEMS = [
    ('single_op',       'What is 37 + 58?',                          '95'),
    ('single_op',       'What is 6 * 9?',                            '54'),
    ('comparison',      'Which is greater: 44 or 51?',               '51'),
    ('number_sequence', 'What is the next number in the sequence: 2, 4, 8, 16, 32, ...?', '64'),
    ('multi_step',      'Calculate: 5 + 3 * 2',                      '11'),
    ('multi_step',      'Calculate: 12 + 8 - 3 * 2',                 '14'),
    ('percentage',      'What is 25% of 80?',                        '20'),
    ('percentage',      'What is 10% of 50?',                        '5'),
    ('ratio',           'Two quantities are in the ratio 3:2. If the total is 50, what is the first quantity?', '30'),
    ('algebra_linear',  'Solve for x: 2x + 4 = 10',                  '3'),
    ('algebra_linear',  'Solve for x: 5x + -3 = -23',                '-4'),
    ('word_problem',    'Alice has 12 apples. She buys 7 more. How many apples does she have now?', '19'),
]

# Check which checkpoints exist
labels = ['init', 'early', 'mid', 'final', 'best']
available = []
for lb in labels:
    path = os.path.join(CK_DIR, f'{lb}.pt')
    exists = os.path.exists(path)
    print(f'  {lb}.pt: {"EXISTS" if exists else "MISSING"}')
    if exists:
        available.append(lb)

In [ ]:
# For each probe problem, run through all available checkpoints
# and record extracted answer + correctness

# Load each checkpoint once, evaluate all problems, release
progression = {}   # label -> {problem -> {extracted, correct, step}}

for label in available:
    ck_path = os.path.join(CK_DIR, f'{label}.pt')
    m, tok, ck_i = load_model_and_tokenizer(ck_path, device=DEVICE)
    step_i = ck_i.get('global_step', '?')
    progression[label] = {'_step': step_i}

    for family, problem, expected in PROBE_PROBLEMS:
        result = generate_answer(
            problem=problem, model=m, tokenizer=tok,
            device=DEVICE, use_reasoning=True, max_new_tokens=64,
        )
        try:
            exp_num = float(expected)
        except ValueError:
            exp_num = 0.0
        vr = verify(result['full_text'], expected, exp_num)
        progression[label][problem] = {
            'extracted': result['extracted_answer'],
            'correct':   vr['correct'],
            'full_text': result['full_text'],
        }

    del m
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'  [{label}] step={step_i} done')

In [ ]:
# Print progression table: rows=problems, cols=checkpoints
col_w = 10
header = f'{"Family":<18} {"Expected":<8}'
for lb in available:
    step_i = progression[lb]['_step']
    header += f'  {lb}(s{step_i})'.ljust(col_w)
print(header)
print('-' * (len(header) + 10))

for family, problem, expected in PROBE_PROBLEMS:
    row = f'{family:<18} {expected:<8}'
    for lb in available:
        d = progression[lb].get(problem, {})
        correct = d.get('correct', False)
        pred    = str(d.get('extracted', '?'))[:8]
        marker  = 'OK' if correct else '--'
        row    += f'  {marker}({pred})'.ljust(col_w)
    print(row)

# Summary: correct count per checkpoint
print()
print('Correct per checkpoint:')
for lb in available:
    n_correct = sum(
        1 for _, prob, _ in PROBE_PROBLEMS
        if progression[lb].get(prob, {}).get('correct', False)
    )
    print(f'  {lb}: {n_correct}/{len(PROBE_PROBLEMS)}')

## 6. Part 5 — Actual Model Outputs (full text)

In [ ]:
# Show full model output for best.pt on every probe problem
# This is the honest view — includes failures, malformed output, correct answers

print('=== Full outputs from best.pt ===')
print()

for family, problem, expected in PROBE_PROBLEMS:
    result = generate_answer(
        problem=problem, model=model, tokenizer=tokenizer,
        device=DEVICE, use_reasoning=True, max_new_tokens=64,
    )
    try:
        exp_num = float(expected)
    except ValueError:
        exp_num = 0.0
    vr = verify(result['full_text'], expected, exp_num)

    status = 'CORRECT' if vr['correct'] else 'WRONG'
    print(f'[{status}] [{family}] expected={expected}, predicted={vr["predicted"]}')
    print(result['full_text'])
    print()

## 7. Part 6 — What the Model Actually Learned

In [ ]:
# Summarise evidence from test benchmark family-level results
print('=== Test accuracy by family (best.pt, n=500 test set) ===')
for fam, acc in sorted(bench.family_accuracy.items(), key=lambda x: -x[1]):
    bar = '#' * int(acc * 30)
    print(f'  {fam:<25} {acc:>6.1%}  {bar}')

print()
print(f'  Overall: {bench.accuracy:.1%}')

# Identify zero-accuracy families
zero_fams = [f for f, a in bench.family_accuracy.items() if a == 0.0]
if zero_fams:
    print(f'\n  Families with 0% accuracy: {zero_fams}')
    print('  These families were in the training data but the model produces no correct answers.')

# Identify high-accuracy families
high_fams = [(f, a) for f, a in bench.family_accuracy.items() if a >= 0.3]
if high_fams:
    print(f'\n  Families >= 30% accuracy: {high_fams}')

In [ ]:
# Check: does the model produce the reasoning label but compute wrong?
# Sample failures from the test benchmark
failures = [s for s in bench.samples if not s['correct']]
print(f'Failure analysis: {len(failures)} wrong answers from first {len(bench.samples)} samples')
print()

# Show 6 failures across different families
shown_families = set()
for s in failures:
    if s['family'] not in shown_families and len(shown_families) < 6:
        shown_families.add(s['family'])
        print(f'[WRONG] [{s["family"]}]')
        print(f'  Problem:   {s["problem"]}')
        print(f'  Expected:  {s["expected"]}')
        print(f'  Predicted: {s["predicted"]}')
        print(f'  Generated text snippet: {s["generated_text"][:150]}')
        print()

# Show 3 correct answers
correct_samples = [s for s in bench.samples if s['correct']]
print(f'Correct answer examples:')
for s in correct_samples[:3]:
    print(f'[CORRECT] [{s["family"]}] {s["problem"]}')
    print(f'  Expected: {s["expected"]} | Predicted: {s["predicted"]}')
    print(f'  Generated: {s["generated_text"][:150]}')
    print()

## 8. Summary Report

In [ ]:
print('=' * 60)
print('EVALUATION SUMMARY — colab_small_baseline / best.pt')
print('=' * 60)
print()
print(f'Model:          {n_params:,} parameters (8m preset, weight-tied)')
print(f'Architecture:   d{cfg.d_model}/L{cfg.n_layers}/H{cfg.n_heads}/ff{cfg.d_ff}')
print(f'Training step:  {step}')
print()
print('Test benchmark (500 unseen medium problems):')
print(f'  Overall:         {bench.correct}/{bench.total} = {bench.accuracy:.1%}')
for fam, acc in sorted(bench.family_accuracy.items(), key=lambda x: -x[1]):
    print(f'  {fam:<25} {acc:.1%}')
print()
print('Generalization benchmark (200 per level, deduplicated from training):')
for level in sorted(gen_results.keys()):
    r = gen_results[level]
    desc = {
        1: 'New values, same templates (medium)',
        2: 'Larger number ranges (hard)',
        3: 'Mixed families (medium)',
        4: 'Long chains, hard (multi_step+algebra)',
        5: 'All families, hard',
    }[level]
    print(f'  Level {level} ({desc}): {r["correct"]}/{r["total"]} = {r["accuracy"]:.1%}')

gen_total_c = sum(r['correct'] for r in gen_results.values())
gen_total_n = sum(r['total']   for r in gen_results.values())
print(f'  Overall generalization: {gen_total_c}/{gen_total_n} = {gen_total_c/gen_total_n:.1%}')
print()
print('Checkpoint progression (correct / 12 probe problems):')
for lb in available:
    n_c = sum(1 for _, prob, _ in PROBE_PROBLEMS
              if progression[lb].get(prob, {}).get('correct', False))
    print(f'  {lb} (step {progression[lb]["_step"]}): {n_c}/12')
print()
print('Families with 0% accuracy (in training data, never solved correctly):')
for fam in zero_fams:
    print(f'  {fam}')
print()
print('NOTE: This model is experimental. 17.8% overall accuracy.')
print('      multi_step and percentage solved 0% of the time.')
print('      number_sequence is the strongest family at ~57%.')
print('      See cell outputs above for actual generated text.')
print('=' * 60)

## 9. Interactive Inference (optional)

In [ ]:
# Uncomment to enter interactive mode
# Type problems, get chain-of-thought output, type 'exit' to stop

# from inference.generate import interactive_inference
# interactive_inference(
#     checkpoint_path=BEST_CK,
#     device=DEVICE,
#     max_new_tokens=128,
#     temperature=0.0,
#     verify_answers=True,   # prompts for expected answer to run verifier
# )